In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab2 ETL Raw to Star") \
    .config("spark.jars", "/home/jovyan/work/jars/postgresql-42.7.3.jar") \
    .getOrCreate()

In [2]:
jdbc_url = "jdbc:postgresql://postgres:5432/lab2_db"

connection_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

In [3]:
raw_df = spark.read.jdbc(
    url=jdbc_url,
    table="mock_data",
    properties=connection_properties
)

In [4]:
raw_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_postal_code: string (nullable = true)
 |-- customer_pet_type: string (nullable = true)
 |-- customer_pet_name: string (nullable = true)
 |-- customer_pet_breed: string (nullable = true)
 |-- seller_first_name: string (nullable = true)
 |-- seller_last_name: string (nullable = true)
 |-- seller_email: string (nullable = true)
 |-- seller_country: string (nullable = true)
 |-- seller_postal_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_price: decimal(10,2) (nullable = true)
 |-- product_quantity: integer (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- sale_customer_id: integer (nullable = true)


In [5]:
raw_df.show(5, truncate=False)

+---+-------------------+------------------+------------+------------------------+----------------+--------------------+-----------------+-----------------+------------------+-----------------+----------------+------------------------+--------------+------------------+------------+----------------+-------------+----------------+----------+----------------+--------------+---------------+-------------+----------------+----------+--------------+----------+-----------+-------------+------------+-----------------------------------+------------+--------------+-------------+------------+-------------+----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
raw_df.count()

10000

In [7]:
from pyspark.sql.functions import col, trim, lower, to_date
from pyspark.sql.types import IntegerType, DecimalType

In [8]:
clean_df = (
    raw_df
    .withColumn("customer_email", trim(lower(col("customer_email"))))
    .withColumn("seller_email", trim(lower(col("seller_email"))))
    .withColumn("store_email", trim(lower(col("store_email"))))
    .withColumn("supplier_email", trim(lower(col("supplier_email"))))
    
    .withColumn("sale_date_parsed", to_date(col("sale_date"), "M/d/yyyy"))
    .withColumn("product_release_date_parsed", to_date(col("product_release_date"), "M/d/yyyy"))
    .withColumn("product_expiry_date_parsed", to_date(col("product_expiry_date"), "M/d/yyyy"))
    
    .withColumn("customer_age", col("customer_age").cast(IntegerType()))
    .withColumn("product_quantity", col("product_quantity").cast(IntegerType()))
    .withColumn("sale_quantity", col("sale_quantity").cast(IntegerType()))
    .withColumn("product_reviews", col("product_reviews").cast(IntegerType()))
    
    .withColumn("product_price", col("product_price").cast(DecimalType(10, 2)))
    .withColumn("sale_total_price", col("sale_total_price").cast(DecimalType(10, 2)))
    .withColumn("product_weight", col("product_weight").cast(DecimalType(10, 2)))
    .withColumn("product_rating", col("product_rating").cast(DecimalType(3, 1)))
)

In [9]:
clean_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_postal_code: string (nullable = true)
 |-- customer_pet_type: string (nullable = true)
 |-- customer_pet_name: string (nullable = true)
 |-- customer_pet_breed: string (nullable = true)
 |-- seller_first_name: string (nullable = true)
 |-- seller_last_name: string (nullable = true)
 |-- seller_email: string (nullable = true)
 |-- seller_country: string (nullable = true)
 |-- seller_postal_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_price: decimal(10,2) (nullable = true)
 |-- product_quantity: integer (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- sale_customer_id: integer (nullable = true)


In [10]:
clean_df.select(
    "customer_email",
    "seller_email",
    "store_email",
    "supplier_email",
    "sale_date",
    "sale_date_parsed",
    "product_release_date",
    "product_release_date_parsed",
    "product_expiry_date",
    "product_expiry_date_parsed",
    "product_price",
    "sale_total_price"
).show(10, truncate=False)

+------------------------+-------------------------+-----------------------------------+--------------------------+----------+----------------+--------------------+---------------------------+-------------------+--------------------------+-------------+----------------+
|customer_email          |seller_email             |store_email                        |supplier_email            |sale_date |sale_date_parsed|product_release_date|product_release_date_parsed|product_expiry_date|product_expiry_date_parsed|product_price|sale_total_price|
+------------------------+-------------------------+-----------------------------------+--------------------------+----------+----------------+--------------------+---------------------------+-------------------+--------------------------+-------------+----------------+
|bmassingham0@army.mil   |bmassingham0@answers.com |bmassingham0@networkadvertising.org|bmassingham0@unblog.fr    |5/14/2021 |2021-05-14      |10/19/2011          |2011-10-19             

In [11]:
clean_df.select(
    "sale_date",
    "sale_date_parsed"
).show(10, truncate=False)

+----------+----------------+
|sale_date |sale_date_parsed|
+----------+----------------+
|5/14/2021 |2021-05-14      |
|11/13/2021|2021-11-13      |
|12/4/2021 |2021-12-04      |
|8/10/2021 |2021-08-10      |
|2/4/2021  |2021-02-04      |
|12/16/2021|2021-12-16      |
|7/26/2021 |2021-07-26      |
|3/6/2021  |2021-03-06      |
|6/10/2021 |2021-06-10      |
|1/21/2021 |2021-01-21      |
+----------+----------------+
only showing top 10 rows



In [12]:
clean_df.filter(col("sale_date_parsed").isNull()).count()

0

In [13]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [14]:
customer_window = Window.orderBy("email")

dim_customer = (
    clean_df
    .select(
        col("customer_first_name").alias("first_name"),
        col("customer_last_name").alias("last_name"),
        col("customer_age").alias("age"),
        col("customer_email").alias("email"),
        col("customer_country").alias("country"),
        col("customer_postal_code").alias("postal_code")
    )
    .dropna(subset=["email"])
    .dropDuplicates(["email"])
    .withColumn("customer_id", row_number().over(customer_window))
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "age",
        "email",
        "country",
        "postal_code"
    )
)

In [15]:
dim_customer.printSchema()

root
 |-- customer_id: integer (nullable = false)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- postal_code: string (nullable = true)



In [16]:
dim_customer.show(10, truncate=False)

+-----------+----------+------------+---+---------------------------------+---------+-----------+
|customer_id|first_name|last_name   |age|email                            |country  |postal_code|
+-----------+----------+------------+---+---------------------------------+---------+-----------+
|1          |Valery    |Gamet       |77 |aabelsonm1@networkadvertising.org|China    |NULL       |
|2          |Stillman  |Hambribe    |38 |aabramq6@i2i.jp                  |Colombia |233539     |
|3          |Beatrix   |Dillingstone|18 |aadgerpi@odnoklassniki.ru        |Portugal |4415-400   |
|4          |Anastasia |Ruppert     |64 |aagneaubj@pbs.org                |Portugal |2435-225   |
|5          |Rhianna   |Winsom      |49 |aahrenm7@reddit.com              |Portugal |9950-325   |
|6          |Diego     |Sayre       |61 |aallsep88@meetup.com             |Syria    |NULL       |
|7          |Emelen    |Godding     |47 |aaltoftsdl@cmu.edu               |Indonesia|NULL       |
|8          |Legra  

In [17]:
dim_customer.count()

10000

In [18]:
from pyspark.sql.functions import count

dim_customer.groupBy("email").agg(count("*").alias("cnt")).filter(col("cnt") > 1).show()

+-----+---+
|email|cnt|
+-----+---+
+-----+---+



In [19]:
seller_window = Window.orderBy("email")

dim_seller = (
    clean_df
    .select(
        col("seller_first_name").alias("first_name"),
        col("seller_last_name").alias("last_name"),
        col("seller_email").alias("email"),
        col("seller_country").alias("country"),
        col("seller_postal_code").alias("postal_code")
    )
    .dropna(subset=["email"])
    .dropDuplicates(["email"])
    .withColumn("seller_id", row_number().over(seller_window))
    .select(
        "seller_id",
        "first_name",
        "last_name",
        "email",
        "country",
        "postal_code"
    )
)

In [20]:
dim_seller.printSchema()

root
 |-- seller_id: integer (nullable = false)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- postal_code: string (nullable = true)



In [21]:
dim_seller.show(10, truncate=False)

+---------+-----------+---------+-----------------------------+-------------+-----------+
|seller_id|first_name |last_name|email                        |country      |postal_code|
+---------+-----------+---------+-----------------------------+-------------+-----------+
|1        |Aveline    |Abelson  |aabelsonm1@nymag.com         |Honduras     |NULL       |
|2        |Arlan      |Abram    |aabramq6@ihg.com             |Indonesia    |NULL       |
|3        |Auberon    |Adger    |aadgerpi@drupal.org          |Brazil       |18600-000  |
|4        |Auberon    |Agneau   |aagneaubj@addtoany.com       |Russia       |416325     |
|5        |Arch       |Ahren    |aahrenm7@sitemeter.com       |China        |NULL       |
|6        |Agnola     |Allsep   |aallsep88@opensource.org     |Philippines  |8407       |
|7        |Alexandrina|Altofts  |aaltoftsdl@joomla.org        |Cameroon     |NULL       |
|8        |Asia       |Andrei   |aandreioz@mozilla.org        |Finland      |62501      |
|9        

In [22]:
dim_seller.count()

10000

In [23]:
dim_seller.groupBy("email").count().filter(col("count") > 1).show()

+-----+-----+
|email|count|
+-----+-----+
+-----+-----+



In [24]:
store_window = Window.orderBy("store_email")

dim_store = (
    clean_df
    .select(
        col("store_name"),
        col("store_location"),
        col("store_city"),
        col("store_state"),
        col("store_country"),
        col("store_phone"),
        col("store_email")
    )
    .dropna(subset=["store_email"])
    .dropDuplicates(["store_email"])
    .withColumn("store_id", row_number().over(store_window))
    .select(
        "store_id",
        "store_name",
        "store_location",
        "store_city",
        "store_state",
        "store_country",
        "store_phone",
        "store_email"
    )
)

In [25]:
dim_store.printSchema()

root
 |-- store_id: integer (nullable = false)
 |-- store_name: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- store_city: string (nullable = true)
 |-- store_state: string (nullable = true)
 |-- store_country: string (nullable = true)
 |-- store_phone: string (nullable = true)
 |-- store_email: string (nullable = true)



In [26]:
dim_store.show(10, truncate=False)

+--------+----------+--------------+--------------------+-----------+------------------+------------+-------------------------+
|store_id|store_name|store_location|store_city          |store_state|store_country     |store_phone |store_email              |
+--------+----------+--------------+--------------------+-----------+------------------+------------+-------------------------+
|1       |Wikizz    |Apt 1229      |Binjiang            |NULL       |Uzbekistan        |119-223-5236|aabelsonm1@netscape.com  |
|2       |Plambee   |Suite 62      |Ayapel              |NULL       |Indonesia         |814-621-5640|aabramq6@biglobe.ne.jp   |
|3       |Avamm     |Room 1165     |Grijó               |13         |Thailand          |384-473-0044|aadgerpi@springer.com    |
|4       |Cogidoo   |18th Floor    |Abades              |14         |Colombia          |294-373-0193|aagneaubj@marketwatch.com|
|5       |JumpXS    |Suite 88      |Madalena            |46         |Indonesia         |247-892-9073|aah

In [27]:
dim_store.count()

10000

In [28]:
dim_store.groupBy("store_email").count().filter(col("count") > 1).show()

+-----------+-----+
|store_email|count|
+-----------+-----+
+-----------+-----+



In [29]:
supplier_window = Window.orderBy("supplier_email")

dim_supplier = (
    clean_df
    .select(
        col("supplier_name"),
        col("supplier_contact"),
        col("supplier_email"),
        col("supplier_phone"),
        col("supplier_address"),
        col("supplier_city"),
        col("supplier_country")
    )
    .dropna(subset=["supplier_email"])
    .dropDuplicates(["supplier_email"])
    .withColumn("supplier_id", row_number().over(supplier_window))
    .select(
        "supplier_id",
        "supplier_name",
        "supplier_contact",
        "supplier_email",
        "supplier_phone",
        "supplier_address",
        "supplier_city",
        "supplier_country"
    )
)

In [30]:
dim_supplier.printSchema()

root
 |-- supplier_id: integer (nullable = false)
 |-- supplier_name: string (nullable = true)
 |-- supplier_contact: string (nullable = true)
 |-- supplier_email: string (nullable = true)
 |-- supplier_phone: string (nullable = true)
 |-- supplier_address: string (nullable = true)
 |-- supplier_city: string (nullable = true)
 |-- supplier_country: string (nullable = true)



In [31]:
dim_supplier.show(10, truncate=False)

+-----------+-------------+-------------------+-----------------------------+--------------+----------------+---------------+----------------+
|supplier_id|supplier_name|supplier_contact   |supplier_email               |supplier_phone|supplier_address|supplier_city  |supplier_country|
+-----------+-------------+-------------------+-----------------------------+--------------+----------------+---------------+----------------+
|1          |Yakitri      |Aveline Abelson    |aabelsonm1@stumbleupon.com   |225-676-0570  |12th Floor      |Ojos de Agua   |Jamaica         |
|2          |Rhybox       |Arlan Abram        |aabramq6@amazon.de           |159-596-3798  |Room 1524       |Cinagrog Girang|Sweden          |
|3          |Flashpoint   |Auberon Adger      |aadgerpi@statcounter.com     |510-619-1115  |Suite 84        |Botucatu       |United States   |
|4          |Bubblebox    |Auberon Agneau     |aagneaubj@creativecommons.org|714-491-6929  |PO Box 27188    |Marfino        |China           |

In [32]:
dim_supplier.count()

10000

In [33]:
dim_supplier.groupBy("supplier_email").count().filter(col("count") > 1).show()

+--------------+-----+
|supplier_email|count|
+--------------+-----+
+--------------+-----+



In [34]:
product_window = Window.orderBy(
    "product_name",
    "product_category",
    "product_brand",
    "product_color",
    "product_size"
)

dim_product = (
    clean_df
    .select(
        col("product_name"),
        col("product_category"),
        col("product_price"),
        col("product_quantity"),
        col("product_weight"),
        col("product_color"),
        col("product_size"),
        col("product_brand"),
        col("product_material"),
        col("product_description"),
        col("product_rating"),
        col("product_reviews"),
        col("product_release_date_parsed").alias("product_release_date"),
        col("product_expiry_date_parsed").alias("product_expiry_date"),
        col("pet_category")
    )
    .dropDuplicates([
        "product_name",
        "product_category",
        "product_brand",
        "product_color",
        "product_size"
    ])
    .withColumn("product_id", row_number().over(product_window))
    .select(
        "product_id",
        "product_name",
        "product_category",
        "product_price",
        "product_quantity",
        "product_weight",
        "product_color",
        "product_size",
        "product_brand",
        "product_material",
        "product_description",
        "product_rating",
        "product_reviews",
        "product_release_date",
        "product_expiry_date",
        "pet_category"
    )
)

In [35]:
dim_product.printSchema()

root
 |-- product_id: integer (nullable = false)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_price: decimal(10,2) (nullable = true)
 |-- product_quantity: integer (nullable = true)
 |-- product_weight: decimal(10,2) (nullable = true)
 |-- product_color: string (nullable = true)
 |-- product_size: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_material: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- product_rating: decimal(3,1) (nullable = true)
 |-- product_reviews: integer (nullable = true)
 |-- product_release_date: date (nullable = true)
 |-- product_expiry_date: date (nullable = true)
 |-- pet_category: string (nullable = true)



In [36]:
dim_product.show(10, truncate=False)

+----------+------------+----------------+-------------+----------------+--------------+-------------+------------+-------------+----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+---------------+--------------------+-------------------+------------+
|product_id|product_name|product_category|product_price|product_quantity|product_weight|product_color|product_size|product_brand|product_material|product_description                                                                                                                                                                                                                            

In [37]:
dim_product.count()

9742

In [38]:
dim_product.groupBy(
    "product_name",
    "product_category",
    "product_brand",
    "product_color",
    "product_size"
).count().filter(col("count") > 1).show()

+------------+----------------+-------------+-------------+------------+-----+
|product_name|product_category|product_brand|product_color|product_size|count|
+------------+----------------+-------------+-------------+------------+-----+
+------------+----------------+-------------+-------------+------------+-----+



In [39]:
from pyspark.sql.functions import dayofmonth, month, year, quarter

In [40]:
date_window = Window.orderBy("full_date")

dim_date = (
    clean_df
    .select(col("sale_date_parsed").alias("full_date"))
    .dropna(subset=["full_date"])
    .dropDuplicates(["full_date"])
    .withColumn("day", dayofmonth(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("year", year(col("full_date")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("date_id", row_number().over(date_window))
    .select(
        "date_id",
        "full_date",
        "day",
        "month",
        "year",
        "quarter"
    )
)

In [41]:
dim_date.printSchema()

root
 |-- date_id: integer (nullable = false)
 |-- full_date: date (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)



In [42]:
dim_date.show(10, truncate=False)

+-------+----------+---+-----+----+-------+
|date_id|full_date |day|month|year|quarter|
+-------+----------+---+-----+----+-------+
|1      |2021-01-01|1  |1    |2021|1      |
|2      |2021-01-02|2  |1    |2021|1      |
|3      |2021-01-03|3  |1    |2021|1      |
|4      |2021-01-04|4  |1    |2021|1      |
|5      |2021-01-05|5  |1    |2021|1      |
|6      |2021-01-06|6  |1    |2021|1      |
|7      |2021-01-07|7  |1    |2021|1      |
|8      |2021-01-08|8  |1    |2021|1      |
|9      |2021-01-09|9  |1    |2021|1      |
|10     |2021-01-10|10 |1    |2021|1      |
+-------+----------+---+-----+----+-------+
only showing top 10 rows



In [43]:
dim_date.count()

364

In [44]:
dim_date.groupBy("full_date").count().filter(col("count") > 1).show()

+---------+-----+
|full_date|count|
+---------+-----+
+---------+-----+



In [45]:
f = clean_df.alias("f")
c = dim_customer.alias("c")
s = dim_seller.alias("s")
st = dim_store.alias("st")
sp = dim_supplier.alias("sp")
p = dim_product.alias("p")
d = dim_date.alias("d")

In [46]:
fact_sales = (
    f
    .join(c, col("f.customer_email") == col("c.email"), "left")
    .join(s, col("f.seller_email") == col("s.email"), "left")
    .join(st, col("f.store_email") == col("st.store_email"), "left")
    .join(sp, col("f.supplier_email") == col("sp.supplier_email"), "left")
    .join(
        p,
        (col("f.product_name") == col("p.product_name")) &
        (col("f.product_category") == col("p.product_category")) &
        (col("f.product_brand") == col("p.product_brand")) &
        (col("f.product_color") == col("p.product_color")) &
        (col("f.product_size") == col("p.product_size")),
        "left"
    )
    .join(d, col("f.sale_date_parsed") == col("d.full_date"), "left")
    .select(
        col("d.date_id").alias("date_id"),
        col("c.customer_id").alias("customer_id"),
        col("s.seller_id").alias("seller_id"),
        col("p.product_id").alias("product_id"),
        col("st.store_id").alias("store_id"),
        col("sp.supplier_id").alias("supplier_id"),
        col("f.sale_quantity").alias("sale_quantity"),
        col("f.sale_total_price").alias("sale_total_price")
    )
)

In [47]:
fact_sales.printSchema()

root
 |-- date_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- seller_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- supplier_id: integer (nullable = true)
 |-- sale_quantity: integer (nullable = true)
 |-- sale_total_price: decimal(10,2) (nullable = true)



In [48]:
fact_sales.show(10, truncate=False)

+-------+-----------+---------+----------+--------+-----------+-------------+----------------+
|date_id|customer_id|seller_id|product_id|store_id|supplier_id|sale_quantity|sale_total_price|
+-------+-----------+---------+----------+--------+-----------+-------------+----------------+
|207    |5424       |5424     |3645      |5424    |5424       |1            |444.84          |
|302    |3034       |3034     |7183      |3034    |3034       |4            |116.90          |
|350    |2570       |2570     |1703      |2570    |2570       |3            |284.66          |
|134    |1199       |1199     |8276      |1199    |1199       |4            |487.70          |
|35     |8514       |8514     |1074      |8514    |8514       |6            |72.27           |
|338    |9531       |9531     |1765      |9531    |9531       |9            |144.24          |
|65     |9358       |9358     |12        |9358    |9358       |2            |127.97          |
|21     |5112       |5112     |9413      |5112    

In [49]:
fact_sales.count()

10000

In [50]:
from pyspark.sql.functions import sum as spark_sum

fact_sales.select(
    spark_sum(col("date_id").isNull().cast("int")).alias("null_date_id"),
    spark_sum(col("customer_id").isNull().cast("int")).alias("null_customer_id"),
    spark_sum(col("seller_id").isNull().cast("int")).alias("null_seller_id"),
    spark_sum(col("product_id").isNull().cast("int")).alias("null_product_id"),
    spark_sum(col("store_id").isNull().cast("int")).alias("null_store_id"),
    spark_sum(col("supplier_id").isNull().cast("int")).alias("null_supplier_id")
).show()

+------------+----------------+--------------+---------------+-------------+----------------+
|null_date_id|null_customer_id|null_seller_id|null_product_id|null_store_id|null_supplier_id|
+------------+----------------+--------------+---------------+-------------+----------------+
|           0|               0|             0|              0|            0|               0|
+------------+----------------+--------------+---------------+-------------+----------------+



In [51]:
dim_customer.write.jdbc(
    url=jdbc_url,
    table="dim_customer",
    mode="overwrite",
    properties=connection_properties
)

In [52]:
dim_seller.write.jdbc(
    url=jdbc_url,
    table="dim_seller",
    mode="overwrite",
    properties=connection_properties
)

In [53]:
dim_store.write.jdbc(
    url=jdbc_url,
    table="dim_store",
    mode="overwrite",
    properties=connection_properties
)

In [54]:
dim_supplier.write.jdbc(
    url=jdbc_url,
    table="dim_supplier",
    mode="overwrite",
    properties=connection_properties
)

In [55]:
dim_product.write.jdbc(
    url=jdbc_url,
    table="dim_product",
    mode="overwrite",
    properties=connection_properties
)

In [56]:
dim_date.write.jdbc(
    url=jdbc_url,
    table="dim_date",
    mode="overwrite",
    properties=connection_properties
)

In [57]:
fact_sales.write.jdbc(
    url=jdbc_url,
    table="fact_sales",
    mode="overwrite",
    properties=connection_properties
)